# G8 - the width cliff: four candidates, one standard

All four failed: D of 0.62-0.80 against transfer 4.03. Eighth entry in the falsification ledger.

## Storage

In [ ]:
import os
from pathlib import Path
STORAGE   = "drive"
DRIVE_DIR = "/content/drive/MyDrive/convergence_experiment"
LOCAL_DIR = "./convergence_data"
try:
    import google.colab                 # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if STORAGE == "env":
    assert os.environ.get("DATA_DIR"), "STORAGE='env' but DATA_DIR unset"
elif STORAGE == "drive" and IN_COLAB:
    from google.colab import drive; drive.mount("/content/drive")
    os.environ["DATA_DIR"] = DRIVE_DIR
else:
    os.environ["DATA_DIR"] = str(Path(LOCAL_DIR).resolve())
DATA_DIR = Path(os.environ["DATA_DIR"])
print("DATA_DIR:", DATA_DIR)

## Experiment

In [ ]:
# ==========================================================
# REQUIRES G0-EXACT TO HAVE PASSED ITS GATE.
#
# WHY THIS CHANGED. An earlier version of this cell read the spectrum
# from the seven-space RAW reconstruction while comparing against a
# transfer curve measured on the four-space SCALED hub. Those are
# different objects with different spectra, so the four candidates were
# being tested against the wrong one. It now reads hub_exact.npz, whose
# spectrum belongs to the hub the curve came from.
# ==========================================================
# G8 — the width cliff: four candidate statistics, one falsification standard.
# Paste as ONE cell after G7. Needs only the hub spectrum + the G7 sweep.
#
# WHY THIS EXISTS. C.13.11 falsified spectral amplification, but bounded the
# falsification explicitly: only ONE operationalisation was tested — the
# amplification 1/sqrt(lambda) applied to the LAST direction each width
# admits. That statistic rises smoothly and DECELERATES across the cliff
# (1.45x per step before, 1.26x across it), so it cannot explain a
# discontinuity. Three other spectral statistics were named in the report and
# never built. This builds them, plus one non-spectral candidate that G6
# makes available and nobody has tried.
#
# THE STANDARD, PRE-REGISTERED. The same one that killed the first two
# explanations, stated numerically so it cannot be bent afterwards:
#
#   Transfer steps across the sweep:  +0.085, +0.039, +0.015, -0.177
#   The cliff step is 11.8x the magnitude of the step before it, and it
#   reverses sign. That is the shape a candidate must reproduce.
#
#   Discontinuity score, computed on log-ratio steps:
#       D = |dlog stat(512->768)| / mean(|dlog stat| over the three prior steps)
#
#   TWO BANDS, calibrated against controls rather than asserted:
#       D >= 3.0   strong - the statistic breaks where transfer breaks
#       D >= 2.0   FLAG - follow up, do not dismiss
#       D <  2.0   fails; smooth predictor, discontinuous outcome
#
#   The bands come from two synthetic controls run before this cell shipped:
#     NULL     a pure power-law spectrum with no cliff in it scores D = 0.58.
#              Nothing smooth reaches 2.
#     POSITIVE the same spectrum with a genuine 17x jump inserted at 512->768
#              scores only D = 2.33 - because a fast-growing statistic has
#              large log steps everywhere, so a real knee is diluted.
#   Transfer itself scores D = 4.03, but transfer is bounded and decelerating,
#   which flatters it. A 3.0 cut alone would therefore MISS a real mechanism
#   carried by a fast-growing statistic. Hence the 2.0 flag band: this is a
#   SCREEN, not a proof, and it is calibrated to fail safe in both directions.
#
#   NOTE ON THE METRIC, and why it is log-ratio rather than absolute.
#   A first version of this cell used absolute step sizes. On a synthetic
#   power-law spectrum with NO cliff in it, accumulated noise energy scored
#   D = 3.2 and "passed" - because an exponentially growing statistic always
#   has its largest absolute step last, whatever the transfer curve does.
#   Absolute deltas are not comparable between a quantity bounded in [0,1]
#   and one that grows geometrically. Log-ratio steps are scale-free, give
#   D ~ 1 for any pure power law, and are the same form the report already
#   used to kill amplification (1.45x per step before, 1.26x across - those
#   are growth RATIOS). Statistics that can be zero or negative fall back to
#   min-max normalisation over the sweep.
#
#   A monotone, smoothly-accelerating statistic scores near 1.0 and FAILS,
#   no matter how appealing the story attached to it.
#
#   Anything with D between 2 and 3 is NOT a pass. Record it as suggestive,
#   name it as untested, and move on. The project has seven falsified
#   explanations because it did not round these up.
#
# HONEST PRIOR: the most likely outcome is that all four fail and the cliff
# stays unexplained. That is a publishable line, already written in C.13.11.
# This cell is worth running because it is cheap and closes a named gap —
# not because a result is expected.
# ==========================================================
import os
import numpy as np
from pathlib import Path

DATA_DIR = Path(os.environ.get("DATA_DIR", "."))
WIDTHS = np.array([64, 128, 256, 512, 768])
TRANSFER = np.array([0.350, 0.435, 0.474, 0.489, 0.312])   # G7 zero-shot
N_TRAIN = 8533
PASS_D, FLAG_D = 3.0, 2.0

# --- adjust to your artifact names ------------------------------------
HUB_NPZ = DATA_DIR / "hub_exact.npz"   # written by G0-exact
# ----------------------------------------------------------------------

hub = np.load(HUB_NPZ)
S = np.asarray(hub["sv"], dtype=np.float64)    # singular values, descending
assert S.ndim == 1 and len(S) >= WIDTHS.max(), (
    f"need at least {WIDTHS.max()} singular values, have {len(S)}")
lam = S ** 2 / N_TRAIN

# sanity against the published table (Math_and_Terms):
# d=512 -> S 124.49, 1/sqrt(lambda) 0.74 ; d=768 -> S 98.92, 0.93
print("spectrum check - SHAPE, after fitting one global scale.")
print("Absolute sigma is not the test: whitening divides by sqrt(lambda),")
print("so a global rescaling of the input cancels exactly in the hub")
print("coordinates. Only the ratios between widths carry information.")
_pub = np.array([376.82, 260.42, 181.61, 124.49, 98.92])
_ws = [64, 128, 256, 512, 768]
_ours = np.array([S[d-1] for d in _ws])
_c = float(np.exp(np.mean(np.log(_pub[1:] / _ours[1:]))))
_res = (_c * _ours) / _pub - 1.0
for i, d in enumerate(_ws):
    print(f"  d={d:4d}  S={_ours[i]:8.2f}   scaled {_c*_ours[i]:8.2f}   "
          f"published {_pub[i]:8.2f}   {_res[i]:+.1%}")
print(f"  global scale {_c:.4f}, shape error {np.abs(_res[1:]).max():.2%} "
      f"across widths 128-768")

In [ ]:
# ---------- the falsification standard ----------
def discontinuity(vals):
    """Scale-free discontinuity score.

    D = |dlog(last step)| / mean|dlog(prior steps)|.

    Log-ratio, not absolute difference: an absolute-delta version scores 3.2
    on a smooth power law with no cliff in it, because a geometrically
    growing statistic always takes its biggest absolute step last. Log steps
    are constant for a pure power law, so that case correctly returns ~1.
    Falls back to min-max normalisation where values are not strictly
    positive.
    """
    v = np.asarray(vals, dtype=np.float64)
    if np.all(v > 0):
        d = np.abs(np.diff(np.log(v)))
    else:
        rng_ = v.max() - v.min()
        d = np.abs(np.diff(v / rng_)) if rng_ > 0 else np.zeros(len(v) - 1)
    return float(d[-1] / (d[:-1].mean() + 1e-12))

D_transfer = discontinuity(TRANSFER)
print(f"\ntransfer D = {D_transfer:.2f}   bands: flag >= {FLAG_D}, "
      f"strong >= {PASS_D}")
print("  metric is log-ratio, not absolute - see the note in the header cell")

In [ ]:
# ---------- the noise threshold: Marchenko-Pastur bulk edge ----------
# For an n x p matrix of iid noise with variance sigma^2 and aspect
# gamma = p/n, eigenvalues below sigma^2 (1 + sqrt(gamma))^2 are
# indistinguishable from noise. Estimate sigma^2 from the spectrum tail.
def mp_edge(p, n, lam_all):
    gamma = p / n
    sigma2 = np.median(lam_all[int(0.75 * len(lam_all)):])   # tail median
    return sigma2 * (1 + np.sqrt(gamma)) ** 2

In [ ]:
# ---------- CANDIDATE A: accumulated noise energy ----------
# Total amplification mass admitted, summed over noise-dominated directions.
# Named in C.13.11 and never built.
A_vals = []
for d in WIDTHS:
    tau = mp_edge(d, N_TRAIN, lam[:d])
    noise = lam[:d][lam[:d] < tau]
    A_vals.append(float(np.sum(1.0 / np.sqrt(noise))) if len(noise) else 0.0)

In [ ]:
# ---------- CANDIDATE B: noise-to-signal direction ratio ----------
B_vals = []
for d in WIDTHS:
    tau = mp_edge(d, N_TRAIN, lam[:d])
    n_noise = int((lam[:d] < tau).sum())
    n_sig = d - n_noise
    B_vals.append(n_noise / max(n_sig, 1))

In [ ]:
# ---------- CANDIDATE C: condition number ----------
C_vals = [float(lam[0] / lam[d - 1]) for d in WIDTHS]

In [ ]:
# ---------- CANDIDATE D: shared vs private hub directions ----------
# NOT spectral, and not named in the report. G6 established that spaces share
# as few as 6 of 64 CCA directions cross-modally, and that intrinsic dimension
# is 11-20 per space. So the shared subspace is SMALL. The hypothesis:
# past some width the hub stops admitting directions that all encoders
# populate and starts admitting directions PRIVATE to one encoder. A head
# trained on encoder A then reads coordinates that carry no signal in B —
# and that flip has a threshold by construction, which is the shape the two
# falsified candidates lacked.
#
# Operationalised: for each hub direction j, compute each encoder's share of
# the variance along it. A direction is SHARED if the minimum share across
# encoders exceeds 1/(2*n_enc) — i.e. every encoder meaningfully populates it.
try:
    # four spaces here, not seven: the hub the transfer curve was measured
    # on. The earlier seven-space version made "shared by all encoders" a
    # harder condition than the original hub ever imposed.
    names = [str(x) for x in hub["space_names"]]
    Hs = np.stack([hub[f"space_{n}"][hub["train_idx"]] @ hub[f"to_hub_{n}"]
                   for n in names])
    n_enc = Hs.shape[0]
    print(f"  candidate D over {n_enc} spaces: {', '.join(names)}")
    var = np.stack([H.var(axis=0) for H in Hs])          # [n_enc, HUB_DIM_MAX]
    share = var / (var.sum(axis=0, keepdims=True) + 1e-12)
    min_share = share.min(axis=0)                        # [HUB_DIM_MAX]
    thresh = 1.0 / (2 * n_enc)
    D_vals = [float((min_share[:d] >= thresh).sum()) / d for d in WIDTHS]
    have_D = True
except KeyError as e:
    print(f"\n  NOTE: hub artifact incomplete ({e}). Candidate D skipped - "
          "re-run G0-exact, which writes the per-space maps it needs.")
    D_vals, have_D = [np.nan] * len(WIDTHS), False

In [ ]:
# ---------- read them ----------
cands = [
    ("A  accumulated noise energy  sum 1/sqrt(lam)", A_vals),
    ("B  noise/signal direction ratio", B_vals),
    ("C  condition number  lam_1/lam_d", C_vals),
]
if have_D:
    cands.append(("D  fraction of hub dirs SHARED by all encoders", D_vals))

print("\n" + "=" * 72)
print(f"{'statistic':<46}" + "".join(f"{w:>8d}" for w in WIDTHS) + "      D")
print("=" * 72)
print(f"{'zero-shot transfer (the target)':<46}"
      + "".join(f"{v:>8.3f}" for v in TRANSFER) + f"  {D_transfer:>6.2f}")
print("-" * 72)
for name, vals in cands:
    d = discontinuity(vals)
    print(f"{name:<46}" + "".join(f"{v:>8.2f}" for v in vals) + f"  {d:>6.2f}")

print("\n" + "=" * 72)
scored = [(n, discontinuity(v)) for n, v in cands]
passed = [(n, d) for n, d in scored if d >= FLAG_D]
if not passed:
    print("VERDICT: all candidates FAIL the discontinuity standard.")
    print("The cliff remains UNEXPLAINED. C.13.11 stands as written, and its")
    print("boundary clause can now be narrowed: four operationalisations")
    print("tested, not one. Report the negative - it is the eighth entry in")
    print("the falsification ledger and it cost one cell.")
else:
    for n, d in passed:
        band = "STRONG" if d >= PASS_D else "FLAG - follow up"
        print(f"{band}: {n}  (D = {d:.2f})")
    print("\nBefore believing it, three checks - a single passing statistic")
    print("out of four is exactly what multiple comparisons produce:")
    print("  1. Does it hold at a second hub seed and a second reference")
    print("     encoder? A threshold that moves with the seed is noise.")
    print("  2. Does it PREDICT out of sample - insert width 640 and 896,")
    print("     predict transfer before measuring, then measure.")
    print("  3. Is the direction right? The statistic must jump where")
    print("     transfer FALLS, not merely change fast somewhere.")
    print("Until all three pass, this is a hypothesis, not an explanation.")
    print("The project's own preferred account already failed once here.")

print("\nEither way: this closes 'other spectral statistics for the cliff',")
print("which was named-but-not-built. It does not need to succeed to close it.")